In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import gradio as gr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingRegressor, StackingRegressor
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

Data **Loading**

In [ ]:
df = pd.read_csv("diabetes.csv")
df.head()


In [ ]:
df.shape

In [ ]:
!pip install ydata-profiling


In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport( df , title="Diabetes prediction", explorative = True  )

profile.to_file("ydata.html")

Data Preprocessing

In [ ]:
print(df.isnull().sum())

In [ ]:
columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in columns:
    df[col] = df[col].replace(0, df[col].mean())

In [ ]:
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)

IQR = Q3 - Q1

outliers = ((df < (Q1 - 1.5 * IQR)) |
            (df > (Q3 + 1.5 * IQR)))

print(outliers.sum())

In [ ]:
x = df.drop("Outcome", axis=1)
y = df["Outcome"]


X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)
print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

**Pipeline**

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print(y_pred)

primary model selection

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(y_pred)

logistic regression was selected because the dataset represents a binary classification problem (diabetic or non diabetic).it is simple ,fast,iterpretable and perfoms well on medical datasets.the model also works efficiently after feature scaling and provides good basline accuracy.

cross-validation

In [ ]:
cv_score = cross_val_score(pipeline,X_train,y_train,cv=5,scoring='accuracy')
print(cv_score)
print(cv_score.mean())
print(cv_score.std())

Hyperparameter Tuning

In [ ]:
param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__solver': ['liblinear', 'lbfgs']
}

grid_search = GridSearchCV(
                           pipeline,
                           param_grid,
                           cv=5,
                           scoring='accuracy',
                           n_jobs = -1
                           )

grid_search.fit(X_train, y_train)

print("best parameters :",grid_search.best_params_)
print("best cv score : ",grid_search.best_score_)

best model selection

In [ ]:
best_model = grid_search.best_estimator_
print(best_model)

model performance evaluation

In [ ]:
y_pred = best_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

confusion_matrix = confusion_matrix(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

print("mean Squared Error:", mse)
print("mean Absolute Error:", mae)
print("r-squared:", r2)
print("confusion Matrix:\n", confusion_matrix)
print("classification Report:\n", classification_rep)

web interface with gradio

In [ ]:
!pip install gradio

In [ ]:
def predict_diabetes(Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age):
    input_data = np.array([[Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age]])
    prediction = best_model.predict(input_data)
    if prediction[0] == 1:
        return "the person is predicted to have diabetes."
    else:
        return "the person is predicted to not have diabetes."

interface = gr.Interface(
    fn=predict_diabetes,
    inputs=[
        gr.Number(label="Pregnancies"),
        gr.Number(label="Glucose"),
        gr.Number(label="BloodPressure"),
        gr.Number(label="SkinThickness"),
        gr.Number(label="Insulin"),
        gr.Number(label="BMI"),
        gr.Number(label="DiabetesPedigreeFunction"),
        gr.Number(label="Age")
    ],
    outputs=gr.Textbox(label="prediction"),
    title="diabetes prediction",
    description="enter the patient's information to predict diabetes."
)

interface.launch(share=True )

In [ ]:
import pickle

with open("diabetes_model.pkl", "wb") as file:
    pickle.dump(best_model, file)

print("model saved")

In [ ]:
app_code = """
import gradio as gr
import pickle
import numpy as np

with open("diabetes_model.pkl", "rb") as file:
    model = pickle.load(file)

def predict_diabetes(Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age):
    input_data = np.array([[Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age]],dtype=float)

    prediction = model.predict(input_data)

    if prediction[0] == 1:
        return "the person is predicted to have diabetes."
    else:
        return "the person is predicted to not have diabetes."



interface = gr.Interface(
    fn=predict_diabetes,
    inputs=[
        gr.Number(label="Pregnancies"),
        gr.Number(label="Glucose"),
        gr.Number(label="BloodPressure"),
        gr.Number(label="SkinThickness"),
        gr.Number(label="Insulin"),
        gr.Number(label="BMI"),
        gr.Number(label="DiabetesPedigreeFunction"),
        gr.Number(label="Age")
        ],

        outputs=gr.Textbox(label="prediction"),
        title="diabetes prediction",

        description="enter the patient's information to predict diabetes."
)

interface.launch()
"""


with open("app.py", "w") as file:
    file.write(app_code)

print("app.py created")



In [ ]:
requirements = """

gradio
numpy
scikit-learn
pandas
ofiling
joblib
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created")